# Day 5.2 — Model Configuration and Runtime
Configuration is data; execution is code. If every agent builds its own HTTP request, then
every agent has its own bugs, its own timeout and its own way of hiding a cost.

Here one adapter hides the provider, and the runtime never learns which model it is using.


## Before you begin

### Learning outcomes

- Read a model configuration and build the matching provider adapter.
- Run the same agent through the mock provider (and OpenRouter, if a key is present).
- Read a run's event trace and find where usage and cost would be recorded.

Architecture reference: [Day 5 diagrams D16](../diagrams/source/day_05.md).

### Expected observation

Mock mode completes locally and reports empty usage. A mistyped provider name is rejected before any request is made.


## Concept briefing

## Configuration versus runtime

An agent configuration describes application-specific behavior: instructions, allowed
tools, model settings and limits. The runtime executes that configuration. A research
agent and a safe task agent should use one runtime without sharing inappropriate tools or
permissions.

A provider adapter hides API-specific request and response shapes behind a small
interface. Switching mock, OpenRouter or Ollama should not rewrite policy or the registry.
Provider metadata such as tokens, cost, latency and errors should still be preserved in
events.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/mini_harness"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")

# 3) Day 5 helper: agent configurations are DATA. Read one from configs/<name>.json
#    and turn it into the AgentConfig dataclass the runtime expects.
import json
from mini_harness import AgentConfig, ModelConfig

def load_config(name):
    raw = json.loads((PROJECT_ROOT / "configs" / f"{name}.json").read_text(encoding="utf-8"))
    raw["model"] = ModelConfig(**raw["model"])   # nested dict -> nested dataclass
    return AgentConfig(**raw)

print("Configs      :", sorted(p.stem for p in (PROJECT_ROOT / "configs").glob("*.json")))

## Step 1 — Configuration really is just a file

Nothing clever happens here. We print the raw JSON, then the dataclass it becomes.


In [ ]:
raw = (PROJECT_ROOT / "configs" / "research_agent.json").read_text(encoding="utf-8")
print("--- configs/research_agent.json ---")
print(raw)

research = load_config("research_agent")
print("--- as a Python object ---")
print("type          :", type(research).__name__)
print("model config  :", research.model)
print("temperature   :", research.model.temperature, "(0.0 = as repeatable as the model allows)")
print("max output    :", research.model.max_output_tokens, "tokens")

## Step 2 — One function chooses the adapter

`build_provider` is the only place that knows how each provider is reached. Notice that
we never write `os.environ["OPENROUTER_API_KEY"]`: a missing key means mock mode, not a crash.


In [ ]:
from mini_harness import build_provider

if LIVE:
    # A key is present, so point the SAME config at OpenRouter.
    research.model.provider = "openrouter"
    research.model.model = os.getenv("OPENROUTER_MODEL", "openai/gpt-oss-120b")

provider = build_provider(research.model)
print("Configured provider :", research.model.provider)
print("Adapter object      :", type(provider).__name__)
print("Model name          :", research.model.model)
print()
print("The runtime below is handed this object and never asks what it is.")

## Step 3 — Run it, and fall back if the network misbehaves

Every live call is wrapped. One 400 or one timeout must not stop the class, so we catch
the exception, explain it in one line, and rerun through the mock provider.


In [ ]:
from mini_harness import HarnessRuntime, MockModel, build_demo_registry

registry = build_demo_registry()
runtime = HarnessRuntime(registry, provider)

try:
    result = runtime.run(research, "What does a harness centralise?")
except Exception as exc:                      # noqa: BLE001 - teaching fallback
    print("Live call failed:", type(exc).__name__, exc)
    print("Falling back to the mock provider so the lesson continues.")
    research.model.provider = "mock"
    runtime = HarnessRuntime(registry, MockModel())
    result = runtime.run(research, "What does a harness centralise?")

print("Run id :", result.run_id)
print("Status :", result.status)
print("Output :", result.output)

## Step 4 — The event trace is the same shape either way

This is the payoff of the adapter: swapping providers changes the *content* of events,
never their *shape*. That is what makes runs comparable.


In [ ]:
for event in result.events:
    print(f"{event['event']:<18}", event["details"])

print()
usage = [e["details"]["usage"] for e in result.events if e["event"] == "model_completed"]
print("Usage records attached to model_completed events:", usage)
print("In MOCK mode usage is empty on purpose: nothing was bought, so nothing is counted.")
print("In LIVE mode each record carries prompt_tokens, completion_tokens and cost_usd,")
print("which is how you attribute cost to a whole run rather than to one reply.")

## Step 5 — What belongs where

Three settings, three different homes. Getting this wrong is the most common
configuration bug in agent projects.


In [ ]:
print("temperature, max_output_tokens -> ModelConfig  (how the model writes)")
print("   ->", research.model)
print()
print("max_steps, allowed_tools      -> AgentConfig  (what the agent may do)")
print("   -> max_steps:", research.max_steps, "| allowed_tools:", research.allowed_tools)
print()
print("OPENROUTER_API_KEY            -> the .env file (a secret; never JSON, never a notebook)")
print("   -> key present in this session:", LIVE)

### Try it yourself

Predict what happens if a configuration names a provider that does not exist —
does the harness guess, or does it stop?


In [ ]:
# --- Worked solution ---
# build_provider knows exactly three provider names. Anything else is a
# configuration error, and it is raised BEFORE a request is built or a key is read.
from mini_harness import ModelConfig, build_provider

typo = ModelConfig(provider="openrouetr", model="openai/gpt-oss-120b")  # deliberate typo
try:
    build_provider(typo)
except ValueError as exc:
    print("Rejected:", exc)
    print("-> failing here is good: a typo can never silently become 'no model at all'.")

# The registry and the policy never saw this. They are not part of the provider layer.
print()
print("Registry still holds:", [spec.name for spec in build_demo_registry().discover()])
print("Fix the spelling and the exact same runtime works again:")
print("  ", type(build_provider(ModelConfig(provider="mock"))).__name__)

### Checkpoint

**1. Why does `build_provider` exist instead of the runtime calling `urllib` directly?**

<details><summary>Show answer</summary>

So that the request format of one vendor cannot leak into the loop. The runtime only knows `provider.decide(...) -> ModelDecision`. Swapping mock, OpenRouter or Ollama changes one line of configuration and touches no policy, registry or loop code.

</details>

**2. In mock mode the usage records are empty. Is that a bug?**

<details><summary>Show answer</summary>

No. Mock mode makes no network call and buys no tokens, so there is genuinely nothing to report. The important part is that the *field* exists in every `model_completed` event, so the same reading code works for a live run where the numbers are real.

</details>

### Recap

- Limitation: agents that build their own API requests duplicate bugs, timeouts and cost handling.
- Layer added: a provider adapter behind one `decide()` contract, selected by data.
- Evidence: the same agent config ran through the chosen adapter and produced an identically shaped event trace; a mistyped provider name was refused up front.
